# 04 · Heart-rate track: pulse-lost / dangerous-rhythm detector from PPG

**Track:** heart rate (v4, 2026-11-09). The earlier heart-rate notebook tried to forecast fainting from tilt-table heart rate. Heart rate alone has no usable signature for the common, hypoxic kind of drowning, and the tilt data have no public outcomes; it is archived as `archive/04_hr_syncope_risk_v3_tilt.ipynb`.

The part of drowning where the heart *is* the signal is **cardiac**: rhythm disorders such as long-QT, or dangerous rhythms on cold-water immersion, where the **pulse disappears or becomes chaotic within seconds**. This notebook builds and tests a detector for that from the wearable's **PPG waveform**.

**Training and test data: [PhysioNet/CinC Challenge 2015](https://physionet.org/content/challenge-2015/1.0.0/)** (open, ODC-BY 1.0)
- 750 ICU alarm records, each ending at the moment a bedside monitor raised an alarm.
- **Labels: expert annotators judged each alarm true or false.** A record was kept only when at least two-thirds of annotators agreed. These are the investigators' labels, not ours.
- **Used:** asystole, extreme bradycardia, ventricular tachycardia and ventricular flutter/fibrillation records **that include PPG** (514 records, 123 true).
- **Excluded:** extreme tachycardia, because a heart rate above 140 is normal while swimming; and records without PPG, because the wearable has only PPG.
- **Input:** the 16 s of finger PPG before the alarm, resampled from 250 Hz to the wearable's **100 Hz**. The event starts between 4:50 and 5:00, so the window contains it.

**False-alarm data: WildPPG** (healthy volunteers in daily life, **sternum** PPG at 128 Hz, the same chest site as the wearable). Every alert there is treated as a false alarm. This is a protocol fact (healthy volunteers), not a label we derived.

**Architectures compared** (all ESP32-feasible)

| Name | Type | Trained? |
|---|---|---|
| `rule_definitions` | No pulse for ≥ 4 s **or** 5-beat heart rate < 40 bpm (the Challenge's own asystole and bradycardia definitions, applied to PPG beats) | No: published thresholds, nothing tuned |
| `logistic` | Logistic regression on 10 PPG features | Yes |
| `boosted_trees` | Gradient-boosted trees on the same features | Yes |
| `cnn_1d` | Small 1D convolutional network on the raw 16-s waveform (about 3,600 parameters) | Yes; skipped if PyTorch is unavailable |

**Claims this notebook can support:** how well finger-PPG windows separate expert-confirmed dangerous rhythms from false monitor alarms in ICU patients, and how often each detector alarms on healthy chest PPG during daily life.

**Claims it cannot support:**
- detection of drowning;
- performance on the wearable's own chest MAX30102 signal in water;
- patient-level independence. The Challenge does not publish patient IDs, so a patient may appear in more than one fold.

**Run on Google Colab:** CPU runtime, **Runtime → Run all**.
- **Downloads:** Challenge 2015 (339 MB) is cached on Drive. WildPPG uses notebook 02's verified downloader and cache: about 300 MB per participant, 16 participants.
- **Fewer participants:** set `WILDPPG_IDS`, for example `an0,e61,fex`.

In [ ]:
import importlib.util
required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'joblib': 'joblib',
            'requests': 'requests', 'matplotlib': 'matplotlib', 'IPython': 'ipython'}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ImportError('Missing packages: ' + ', '.join(missing) + '. Install only these with %pip, restart and Run All.')
from pathlib import Path
import hashlib, io, json, math, os, platform, sys, time, zipfile
import numpy as np
import pandas as pd
import requests, joblib, sklearn
from scipy import io as sio, signal, stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from IPython.display import display
HAVE_TORCH = importlib.util.find_spec('torch') is not None

def env_flag(name, default):
    return os.getenv(name, str(default)).lower() in {'1', 'true', 'yes'}

# ---- Storage: Google Colab ----
ON_COLAB = 'google.colab' in sys.modules
SAVE_TO_DRIVE = env_flag('KB_SAVE_TO_DRIVE', ON_COLAB)
if ON_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/KappaBetaML')
else:
    BASE = Path('/content') if ON_COLAB else Path('.')
DATA_ROOT = Path(os.getenv('PULSE_DATA_ROOT', str(BASE / 'input')))
OUT = Path(os.getenv('PULSE_OUTPUT_DIR', str(BASE / 'outputs' / 'pulse_loss')))
FIGURES = OUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.getenv('PULSE_CACHE', str(BASE / 'cache' / 'pulse_loss')))
# Same default cache as notebook 02, so WildPPG is downloaded once per runtime.
WILDPPG_CACHE = Path(os.getenv('WILDPPG_CACHE', '/content/cache/wildppg_sternum' if ON_COLAB
                               else str(BASE / 'cache' / 'wildppg_sternum')))
WILDPPG_IDS = [s.strip() for s in os.getenv('WILDPPG_IDS', '').split(',') if s.strip()] or None   # None = all 16
ALLOW_DOWNLOAD = env_flag('PULSE_DOWNLOAD', True)
SEED = 20261109

# ---- Fixed for v4 before any result was seen. ----
C2015_FS, DEVICE_FS = 250, 100          # Challenge sampling rate; wearable raw PPG rate
WINDOW_S, ALARM_AT_S = 16, 300          # 16 s of PPG ending at the alarm (event onset 4:50-5:00)
TARGET_ALARMS = ['Asystole', 'Bradycardia', 'Ventricular_Tachycardia', 'Ventricular_Flutter_Fib']
EXCLUDED_ALARMS = {'Tachycardia': 'HR > 140 bpm is normal while swimming'}
ASYSTOLE_GAP_S, BRADY_BPM, BRADY_BEATS = 4.0, 40.0, 5   # Challenge alarm definitions (rule_definitions)
OUTER_FOLDS, INNER_FOLDS = 5, 5
# Two operating points, both declared in advance and chosen on inner (training-only) predictions:
TARGET_SENSITIVITY = 0.90   # 'high_sensitivity': largest threshold with inner sensitivity >= 0.90
TARGET_SPECIFICITY = 0.90   # 'high_specificity': smallest threshold with inner specificity >= 0.90
TRANSFER_STEP_S = 2         # healthy chest PPG scored every 2 s (16-s windows)

def poisson_rate_ci(count, hours, level=0.95):
    a = 1 - level
    lo = 0.0 if count == 0 else stats.chi2.ppf(a / 2, 2 * count) / 2
    return lo / hours, stats.chi2.ppf(1 - a / 2, 2 * count + 2) / 2 / hours

def proportion_ci(k, n, level=0.95):
    if n == 0:
        return np.nan, np.nan
    ci = stats.binomtest(int(k), int(n)).proportion_ci(confidence_level=level, method='wilson')
    return ci.low, ci.high

print('Output:', OUT.resolve(), '| PyTorch available:', HAVE_TORCH)

## Acquire and verify the Challenge 2015 training set

The zip is pinned by MD5 (`99a51fefcd72b584a8122ebd7f78dacf`, 338,623,761 bytes, checked 2026-11-09). A mismatch stops the run.

In [ ]:
C2015 = ('training.zip', 'https://physionet.org/files/challenge-2015/1.0.0/training.zip',
         '99a51fefcd72b584a8122ebd7f78dacf')

def md5(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def acquire(name, url, digest):
    found = list(DATA_ROOT.rglob(name)) if DATA_ROOT.is_dir() else []
    for path in [*found, CACHE / name]:
        if path.is_file() and md5(path) == digest:
            return path
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f'Attach the original {name} below {DATA_ROOT} or enable Internet.')
    CACHE.mkdir(parents=True, exist_ok=True)
    part = CACHE / (name + '.part')
    for attempt in range(4):
        try:
            offset = part.stat().st_size if part.exists() else 0
            headers = {'User-Agent': 'pulse-loss-notebook/1.0', **({'Range': f'bytes={offset}-'} if offset else {})}
            with requests.get(url, stream=True, timeout=(15, 120), headers=headers) as r:
                r.raise_for_status()
                with open(part, 'ab' if offset and r.status_code == 206 else 'wb') as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
            if md5(part) != digest:
                part.unlink()
                raise ValueError('Checksum mismatch; the source changed. Review before use.')
            part.replace(CACHE / name)
            return CACHE / name
        except (requests.RequestException, OSError, ValueError) as error:
            last = error
            time.sleep(2 + 2 * attempt)
    raise RuntimeError(f'Cannot acquire {name}: {last}')

c2015_zip = acquire(*C2015)
print(c2015_zip, f'{c2015_zip.stat().st_size / 1e6:.1f} MB')

## Labels and PPG windows

Each record's label comes from the Challenge's `ALARMS` file (`record,alarm_type,label`; `1` = true alarm, `0` = false alarm), as assigned by the expert annotators. We only select which records to use; we never relabel.

| Alarm | Challenge definition | Used? |
|---|---|---|
| Asystole | No QRS for ≥ 4 s | Yes |
| Extreme bradycardia | HR < 40 bpm for 5 beats | Yes |
| Ventricular tachycardia | ≥ 5 ventricular beats at HR > 100 | Yes |
| Ventricular flutter/fibrillation | Fibrillatory waveform for ≥ 4 s | Yes |
| Extreme tachycardia | HR > 140 bpm for 17 beats | **No**: normal while swimming |

**PPG preparation** (identical for every dataset):
1. Convert to physical units with the header gain.
2. Resample to 100 Hz.
3. Band-pass 0.5–8 Hz.
4. Scale by the window's 5th–95th percentile spread.

The first 4 min 50 s of each record is **not used as "normal"**: it is unlabelled.

In [ ]:
def read_header(z, record):
    lines = z.read(f'training/{record}.hea').decode('latin1').splitlines()
    sigs = []
    for line in lines[1:]:
        if line and not line.startswith('#'):
            f = line.split()
            sigs.append((f[-1], float(f[2].split('/')[0].split('(')[0])))
    return sigs

def to_device_rate(x, fs):
    up, down = DEVICE_FS, fs
    g = math.gcd(up, down)
    return signal.resample_poly(np.asarray(x, float), up // g, down // g)

windows_path = CACHE / 'c2015_ppg_windows.npz'
if windows_path.exists():
    cached = np.load(windows_path, allow_pickle=False)
    X_wave, meta = cached['X'], pd.DataFrame({k: cached[k] for k in ['record', 'alarm', 'label']})
else:
    with zipfile.ZipFile(c2015_zip) as z:
        alarms = pd.read_csv(io.BytesIO(z.read('training/ALARMS')), header=None, names=['record', 'alarm', 'label'])
        waves, rows = [], []
        for r in alarms.itertuples():
            if r.alarm not in TARGET_ALARMS:
                continue
            sigs = read_header(z, r.record)
            names = [n for n, _ in sigs]
            if 'PLETH' not in names:
                continue
            k = names.index('PLETH')
            val = sio.loadmat(io.BytesIO(z.read(f'training/{r.record}.mat')))['val'][k].astype(float) / sigs[k][1]
            seg = val[(ALARM_AT_S - WINDOW_S) * C2015_FS: ALARM_AT_S * C2015_FS]
            waves.append(to_device_rate(seg, C2015_FS)[:WINDOW_S * DEVICE_FS])
            rows.append({'record': r.record, 'alarm': r.alarm, 'label': int(r.label)})
    X_wave, meta = np.vstack(waves).astype(np.float32), pd.DataFrame(rows)
    CACHE.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(windows_path, X=X_wave, record=meta.record.to_numpy(dtype=str),
                        alarm=meta.alarm.to_numpy(dtype=str), label=meta.label.to_numpy(dtype=int))
y = meta.label.to_numpy().astype(int)
display(meta.groupby('alarm').label.agg(records='size', true_alarms='sum').assign(false_alarms=lambda d: d.records - d.true_alarms))
print(f'{len(meta)} records with PPG; {y.sum()} expert-confirmed true events; window {X_wave.shape[1] / DEVICE_FS:.0f} s at {DEVICE_FS} Hz')

## Features and architectures

**PPG features** (one value per 16-s window, from detected pulse peaks):

| Feature | Meaning |
|---|---|
| `max_gap_s` | Longest time without a detected pulse, including the window edges |
| `tail_gap_s` | Time since the last pulse at the end of the window |
| `n_beats` | Number of pulses |
| `hr_median`, `hr_min5` | Median heart rate; lowest 5-beat average heart rate |
| `ibi_cv` | Irregularity of pulse intervals |
| `amp_tail_ratio` | Pulse size in the last 4 s ÷ pulse size in the first 8 s (pulse fading) |
| `spectral_purity` | Share of 0.5–4 Hz power near the dominant pulse frequency (low = noise or chaos) |
| `flat_fraction` | Share of near-flat samples (signal lost) |
| `kurtosis` | Spikiness (motion artefacts) |

`cnn_1d` sees the prepared waveform directly. Three convolution blocks (8, 16, 32 channels) feed global averaging and one output, about 3,600 parameters. It is trained with class weighting and fixed seeds.

In [ ]:
SOS = signal.butter(3, [0.5, 8.0], fs=DEVICE_FS, btype='band', output='sos')
FEATURES = ['max_gap_s', 'tail_gap_s', 'n_beats', 'hr_median', 'hr_min5', 'ibi_cv', 'amp_tail_ratio',
            'spectral_purity', 'flat_fraction', 'kurtosis']

def prepare(x):
    x = signal.sosfiltfilt(SOS, np.asarray(x, float) - np.median(x))
    spread = np.percentile(x, 95) - np.percentile(x, 5)
    return x / spread if spread > 0 else np.zeros_like(x)

def ppg_features(raw):
    z = prepare(raw)
    duration = len(z) / DEVICE_FS
    if not np.any(z):
        return dict(zip(FEATURES, [duration, duration, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0])), np.array([], int)
    peaks, prop = signal.find_peaks(z, distance=int(0.3 * DEVICE_FS), prominence=0.3)
    t = peaks / DEVICE_FS
    gaps = np.diff(np.r_[0.0, t, duration])
    ibi = np.diff(t)
    hr = 60 / ibi if len(ibi) else np.array([0.0])
    hr5 = 60 / pd.Series(ibi).rolling(BRADY_BEATS).mean().dropna().to_numpy() if len(ibi) >= BRADY_BEATS else hr
    amps = prop['prominences']
    tail, head = amps[t > duration - 4], amps[t < 8]
    f, p = signal.welch(z, fs=DEVICE_FS, nperseg=min(len(z), 512))
    band = (f >= 0.5) & (f <= 4)
    peak_f = f[band][np.argmax(p[band])]
    purity = p[band][np.abs(f[band] - peak_f) <= 0.25].sum() / max(p[band].sum(), 1e-12)
    values = [gaps.max(), gaps[-1], len(peaks), float(np.median(hr)), float(np.min(hr5)),
              float(np.std(ibi) / np.mean(ibi)) if len(ibi) > 1 else 0.0,
              float(np.median(tail) / np.median(head)) if len(tail) and len(head) else 0.0,
              float(purity), float((np.abs(np.diff(z)) < 0.002).mean()), float(stats.kurtosis(z))]
    return dict(zip(FEATURES, values)), peaks

def rule_definitions(F):
    # Challenge alarm definitions applied to PPG pulses: no pulse >= 4 s, or 5-beat HR < 40 bpm.
    return ((F['max_gap_s'] >= ASYSTOLE_GAP_S) | ((F['hr_min5'] < BRADY_BPM) & (F['n_beats'] > 0))).astype(float).to_numpy()

feature_table = pd.DataFrame([ppg_features(w)[0] for w in X_wave])
X_feat = feature_table[FEATURES].to_numpy(float)
X_norm = np.vstack([prepare(w) for w in X_wave]).astype(np.float32)

def balanced_log_loss(labels, p):
    # Mean of the per-class mean log losses. Every learned model here is trained on a class-weighted log loss
    # (class_weight='balanced' or pos_weight = negatives/positives), which is proportional to this quantity,
    # so train, validation and test values are on the same scale as the training objective.
    labels, p = np.asarray(labels), np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7)
    parts = [-np.log(p[labels == 1]).mean() if (labels == 1).any() else np.nan,
             -np.log(1 - p[labels == 0]).mean() if (labels == 0).any() else np.nan]
    return float(np.nanmean(parts))

def split_scores(labels, p, threshold=0.5):
    # Loss plus accuracy at 0.5, the natural decision point of a class-balanced model (not an operating point).
    labels, flag = np.asarray(labels), np.asarray(p) >= threshold
    se = flag[labels == 1].mean() if (labels == 1).any() else np.nan
    sp = (~flag[labels == 0]).mean() if (labels == 0).any() else np.nan
    return {'loss': balanced_log_loss(labels, p), 'accuracy': float((flag == (labels == 1)).mean()),
            'balanced_accuracy': float(np.nanmean([se, sp]))}

def make_feature_model(name):
    if name == 'logistic':
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=20,
                                          l2_regularization=1.0, class_weight='balanced', random_state=SEED)

if HAVE_TORCH:
    import torch
    from torch import nn
    torch.set_num_threads(min(4, os.cpu_count() or 1))

    class PulseCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.body = nn.Sequential(
                nn.Conv1d(1, 8, 7, stride=2, padding=3), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(8, 16, 7, stride=2, padding=3), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
            self.head = nn.Linear(32, 1)
        def forward(self, x):
            return self.head(self.body(x.unsqueeze(1)).squeeze(-1)).squeeze(-1)

    class CNNModel:
        def __init__(self, epochs=40, seed=SEED):
            self.epochs, self.seed = epochs, seed
        def fit(self, X, y, monitor=None):
            # monitor: optional {split: (X, y)} scored after every epoch. Scoring uses no randomness and no gradients,
            # so the trained weights are identical with or without it.
            torch.manual_seed(self.seed)
            rng = np.random.default_rng(self.seed)
            self.net = PulseCNN()
            self.history = []
            opt = torch.optim.Adam(self.net.parameters(), lr=1e-3, weight_decay=1e-3)
            pos = max(1, int(y.sum()))
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(y) - pos) / pos, dtype=torch.float32))
            Xt, yt = torch.tensor(X), torch.tensor(y, dtype=torch.float32)
            for epoch in range(self.epochs):
                self.net.train()
                order = rng.permutation(len(y))
                for b in range(0, len(order), 32):
                    idx = order[b:b + 32]
                    gain = torch.tensor(rng.uniform(0.8, 1.2, (len(idx), 1)), dtype=torch.float32)  # amplitude jitter
                    opt.zero_grad()
                    loss_fn(self.net(Xt[idx] * gain), yt[idx]).backward()
                    opt.step()
                for split, (Xm, ym) in (monitor or {}).items():
                    self.history.append({'step': epoch + 1, 'split': split, **split_scores(ym, self.predict_proba(Xm)[:, 1])})
            return self
        def predict_proba(self, X):
            self.net.eval()
            with torch.no_grad():
                p = torch.sigmoid(self.net(torch.tensor(np.asarray(X, np.float32)))).numpy()
            return np.c_[1 - p, p]
    print('cnn_1d parameters:', sum(p.numel() for p in PulseCNN().parameters()))

MODELS = ['rule_definitions', 'logistic', 'boosted_trees'] + (['cnn_1d'] if HAVE_TORCH else [])

def fit(name, idx, monitor=None):
    '''monitor: optional {split: record indices}. Loss and accuracy are recorded per CNN epoch or per boosting round.
    The curves are diagnostics only: 40 epochs and 200 rounds were fixed in advance, and nothing is chosen from them.'''
    monitor = monitor or {}
    if name == 'cnn_1d':
        return CNNModel().fit(X_norm[idx], y[idx], {s: (X_norm[i], y[i]) for s, i in monitor.items()})
    model = make_feature_model(name).fit(X_feat[idx], y[idx])
    model.history = []
    if name == 'boosted_trees':   # logistic regression is solved to convergence: one solution, no training curve
        for split, i in monitor.items():
            for step, p in enumerate(model.staged_predict_proba(X_feat[i]), 1):
                model.history.append({'step': step, 'split': split, **split_scores(y[i], p[:, 1])})
    return model

def predict(name, model, idx):
    if name == 'rule_definitions':
        return rule_definitions(feature_table.iloc[idx])
    X = X_norm[idx] if name == 'cnn_1d' else X_feat[idx]
    return model.predict_proba(X)[:, 1]
print('Architectures:', MODELS)

## Evaluation protocol

- **Outer 5-fold cross-validation**, stratified by alarm type × label. Each record is scored once by models that never saw it. Patient IDs are not published, so folds are record-level.
- **Thresholds come only from training data.** Inside each outer training fold, a 5-fold inner cross-validation produces training-only scores. Two operating points are then chosen:
  - `high_sensitivity`: the largest threshold with inner sensitivity ≥ 0.90;
  - `high_specificity`: the smallest threshold with inner specificity ≥ 0.90.

  `rule_definitions` has no threshold.
- **Reported:**
  - AUROC with a bootstrap 95% CI;
  - sensitivity and specificity at each operating point, with Wilson 95% CIs;
  - per-alarm-type results;
  - accuracy and balanced accuracy (the mean of sensitivity and specificity). 76% of records are false alarms, so a detector that never fires already has 0.76 accuracy; read balanced accuracy alongside it;
  - the Challenge score, (TP + TN) / (TP + TN + FP + 5·FN), which counts a missed event five times as heavily as a false alarm.
- **Training diagnostics** (next section) are recorded from the same fitted models, so they do not change any result.

In [ ]:
def thresholds_from(scores, labels):
    pos, neg = np.sort(scores[labels == 1]), np.sort(scores[labels == 0])
    if not len(pos) or not len(neg):
        return {'high_sensitivity': np.nan, 'high_specificity': np.nan}
    hs = pos[int(np.floor((1 - TARGET_SENSITIVITY) * len(pos)))]   # >= 90% of positives at or above
    hp = neg[min(len(neg) - 1, int(np.ceil(TARGET_SPECIFICITY * len(neg))) - 1)]
    return {'high_sensitivity': float(hs), 'high_specificity': float(np.nextafter(hp, np.inf))}

strata = (meta.alarm + '_' + meta.label.astype(str)).to_numpy()
outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=SEED)
oof = {m: np.full(len(y), np.nan) for m in MODELS}
fold_of = np.full(len(y), -1)
chosen = []
# Diagnostics: train = outer-training records (in-sample), valid = inner held-out folds (the split that chooses
# thresholds), test = outer held-out fold (never seen in fitting or threshold choice).
curves, split_rows, roc_parts = [], [], []
started = time.monotonic()
for fold, (train, test) in enumerate(outer.split(X_feat, strata)):
    fold_of[test] = fold
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED + fold)
    for m in MODELS:
        if m == 'rule_definitions':
            oof[m][test] = predict(m, None, test)
            split_rows.append({'model': m, 'fold': fold, 'split': 'test', 'records': len(test), 'AUROC': np.nan,
                               **split_scores(y[test], oof[m][test]), 'loss': np.nan})   # 0/1 output: no loss
            continue
        inner_scores = np.full(len(train), np.nan)
        for k, (itr, ite) in enumerate(inner.split(X_feat[train], strata[train])):
            model = fit(m, train[itr], monitor={'valid': train[ite]})
            inner_scores[ite] = predict(m, model, train[ite])
            curves += [{'model': m, 'fold': fold, 'inner': k, **h} for h in model.history]
        th = thresholds_from(inner_scores, y[train])
        chosen.append({'fold': fold, 'model': m, **th})
        model = fit(m, train, monitor={'train': train, 'test': test})
        curves += [{'model': m, 'fold': fold, 'inner': -1, **h} for h in model.history]
        oof[m][test] = predict(m, model, test)
        for split, idx, s in [('train', train, predict(m, model, train)), ('valid', train, inner_scores),
                              ('test', test, oof[m][test])]:
            split_rows.append({'model': m, 'fold': fold, 'split': split, 'records': len(idx),
                               'AUROC': roc_auc_score(y[idx], s), **split_scores(y[idx], s)})
            roc_parts.append({'model': m, 'split': split, 'fold': fold, 'labels': y[idx], 'scores': s})
    print(f'outer fold {fold + 1}/{OUTER_FOLDS} done ({time.monotonic() - started:.0f} s)', flush=True)
chosen = pd.DataFrame(chosen)
curves, split_metrics = pd.DataFrame(curves), pd.DataFrame(split_rows)

def bootstrap_auc(scores, labels, n=1000):
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(labels), len(labels))
        if len(np.unique(labels[i])) == 2:
            vals.append(roc_auc_score(labels[i], scores[i]))
    return np.percentile(vals, [2.5, 97.5])

def flags_for(m, point):
    if m == 'rule_definitions':
        return oof[m] >= 0.5
    th = chosen.set_index(['fold', 'model'])[point]
    return np.array([oof[m][i] >= th[(fold_of[i], m)] for i in range(len(y))])

rows, per_type = [], []
for m in MODELS:
    auc = roc_auc_score(y, oof[m]) if m != 'rule_definitions' else np.nan
    auc_lo, auc_hi = bootstrap_auc(oof[m], y) if m != 'rule_definitions' else (np.nan, np.nan)
    for point in (['definition'] if m == 'rule_definitions' else ['high_sensitivity', 'high_specificity']):
        flag = flags_for(m, 'high_sensitivity' if point == 'definition' else point)
        tp, fn = int((flag & (y == 1)).sum()), int((~flag & (y == 1)).sum())
        tn, fp = int((~flag & (y == 0)).sum()), int((flag & (y == 0)).sum())
        se, sp = tp / (tp + fn), tn / (tn + fp)
        rows.append({'model': m, 'operating_point': point, 'AUROC': auc, 'AUROC_lo': auc_lo, 'AUROC_hi': auc_hi,
                     'sensitivity': se, 'sens_lo': proportion_ci(tp, tp + fn)[0], 'sens_hi': proportion_ci(tp, tp + fn)[1],
                     'specificity': sp, 'spec_lo': proportion_ci(tn, tn + fp)[0], 'spec_hi': proportion_ci(tn, tn + fp)[1],
                     'accuracy': (tp + tn) / len(y), 'balanced_accuracy': (se + sp) / 2,
                     'challenge_score': (tp + tn) / (tp + tn + fp + 5 * fn), 'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp})
        for alarm in TARGET_ALARMS:
            a = (meta.alarm == alarm).to_numpy()
            per_type.append({'model': m, 'operating_point': point, 'alarm': alarm,
                             'true_events': int(y[a].sum()), 'caught': int((flag & a & (y == 1)).sum()),
                             'false_alarms': int((a & (y == 0)).sum()), 'wrongly_flagged': int((flag & a & (y == 0)).sum())})
results, per_type = pd.DataFrame(rows), pd.DataFrame(per_type)
display(results.round(3))
display(per_type.pivot_table(index=['model', 'operating_point'], columns='alarm',
                             values=['caught', 'wrongly_flagged'], aggfunc='sum'))

## Training diagnostics: loss, accuracy and ROC on train, validation and test

Every number here comes from the models fitted in the cross-validation above. There are three splits per outer fold:

| Split | Records | Scored by |
|---|---|---|
| **train** | the outer fold's training records (about 411) | the outer-fold model, in-sample |
| **valid** | the same training records, each scored by an inner-CV model that did not see it | the 5 inner-fold models. This is the split that chooses the thresholds |
| **test** | the outer held-out fold (about 103) | the outer-fold model, which never saw these records |

- **Loss** is the class-balanced log loss: the mean of the true-event and false-alarm mean losses. It is proportional to what every learned model minimises during training, so the three splits are directly comparable.
- **Accuracy** and **balanced accuracy** here use a 0.5 cut-off, the natural decision point of a class-balanced model. The operating-point results above use the chosen thresholds instead.
- **Curves** exist only for the iterative models: `boosted_trees` (per boosting round) and `cnn_1d` (per epoch). Logistic regression is solved to a single optimum, so it has only the final values in the table. `rule_definitions` has nothing trained; only its test row is shown.
- **The test curves are shown for transparency only.** The numbers of epochs (40) and boosting rounds (200) were fixed before any result was seen, and nothing is chosen from these curves. Picking a stopping point from the test curve would leak test information into the model.
- **How to read the gap:** a train loss far below the valid and test losses means overfitting. A test value close to valid suggests the thresholds chosen on valid will transfer.

In [ ]:
summary = (split_metrics.groupby(['model', 'split'], sort=False)[['AUROC', 'loss', 'accuracy', 'balanced_accuracy']]
           .agg(['mean', 'min', 'max']))
print('Mean over the 5 outer folds (min-max across folds):')
display(summary.round(3))
gap = split_metrics.pivot_table(index='model', columns='split', values=['AUROC', 'loss'], aggfunc='mean')
gap = gap.dropna(axis=0, how='any')
if len(gap):
    display(pd.DataFrame({'AUROC train - test': gap[('AUROC', 'train')] - gap[('AUROC', 'test')],
                          'loss test - train': gap[('loss', 'test')] - gap[('loss', 'train')]}).round(3))
if len(curves):
    last = curves[curves.step.eq(curves.groupby('model').step.transform('max'))]
    print('Final-step values of the training curves (mean over curves):')
    display(last.groupby(['model', 'split'])[['loss', 'accuracy', 'balanced_accuracy']].mean().round(3))

## False alarms on healthy chest PPG (WildPPG)

The detectors are refitted on all 514 records. Thresholds come from 5-fold cross-validated scores on those records, with the same two operating points. The frozen detectors then run on **sternum infrared PPG** from healthy volunteers going about daily life: a 16-s window every 2 s, resampled from 128 Hz to 100 Hz.
- **Sign:** raw optical intensity *falls* when blood volume rises, so it is negated to match the clinical PLETH convention the detectors were trained on.
- **Label:** the participants are healthy volunteers, so every alert start is counted as a false alarm. The table gives alert starts per hour with exact Poisson 95% intervals.
- **Motion split:** results are also split by measured chest motion (terciles of accelerometer variability within each window). This is descriptive, not a label.
- **Pulse-visibility split:** the chest ECG in the same window gives a reference beat count. Windows where the PPG shows 0.8–1.2× the ECG beats count as "pulse visible". This separates *the detector is wrong* from *the PPG doesn't show the pulse here*.

This is the same chest site as the wearable, on land. Swimming will be harder.

In [ ]:
# The original MATLAB v5 file begins with id, notes and the complete sternum matrix.
# Each manifest entry pins the byte immediately after that matrix and the source ETag.
# Downloading this exact prefix preserves every chest sample at its original precision.
from pathlib import Path
import shutil
import time
from urllib.error import URLError
from urllib.request import Request, urlopen
from scipy.io import whosmat

WILDPPG_BASE = "https://polybox.ethz.ch/public.php/dav/files/NWTuyNojU7aya1y/data/"
STERNUM_FILES = {
    "an0": (295671696, "d185bc1ae93e4c9848524e1aab22a39b"),
    "e61": (296257912, "f88e7a783153615a128013363f2d548e"),
    "fex": (320834792, "7273ac5694d4dd2f91aa264e00ddf00a"),
    "k2s": (365403792, "52134984b7904d0145a99c08d15743bd"),
    "kjd": (321207976, "3eefd5de96a73e60bac2e58db0bc1096"),
    "l38": (319555472, "82658918d9bfd74ae511f227a7224343"),
    "n31": (317742648, "226c271e17ecea4ddb7ed844afc4212f"),
    "ngh": (352395448, "e59df9850310836a42df52caa9012a2b"),
    "p5d": (346957672, "a9128090b639d6c0b97532a12bef7c47"),
    "p9p": (299989752, "b35315f9235f273e6b5363aa43e28a30"),
    "qm9": (315823416, "68ef8616be330628e521e38d75d2eaa4"),
    "ssx": (303828216, "fd7dcc41fc08eb4213e2985580705a75"),
    "trh": (345358264, "f82b2148513c5b9e5924146934e30bab"),
    "tz8": (348397048, "82cc5f445aade45c7d3d6aff8d143577"),
    "u7y": (317955896, "de7445d86d476185a62da918eed15c8e"),
    "w4p": (333469688, "b1a570b7c305275c050467702c9ab2db"),
}

def fetch_verified_range(url, start, end, etag):
    headers = {"Range": f"bytes={start}-{end}", "Accept-Encoding": "identity"}
    request = Request(url, headers=headers)
    for attempt in range(3):
        try:
            with urlopen(request, timeout=120) as response:
                served = response.headers.get("Content-Range", "").split("/")[0]
                if (response.status != 206 or served != f"bytes {start}-{end}"
                        or response.headers.get("ETag", "").strip('"') != etag):
                    raise ValueError("ETH file or HTTP range changed. Attach original WildPPG participant MATs.")
                data = response.read(end - start + 2)
                if len(data) != end - start + 1:
                    raise OSError("Incomplete WildPPG transfer")
                return data
        except (URLError, OSError) as exc:
            if attempt == 2:
                raise RuntimeError("Check the runtime's Internet access or upload the original WildPPG MAT files.") from exc
            time.sleep(2 ** attempt)

def ensure_wildppg_inputs(directory, ids=None):
    """Create or resume the chest recording for each requested participant in a cache."""
    root = Path(directory)
    root.mkdir(parents=True, exist_ok=True)
    selected = list(STERNUM_FILES) if ids is None else list(dict.fromkeys(ids))
    if not selected or set(selected) - STERNUM_FILES.keys():
        raise ValueError(f"Select one or more participant IDs from {list(STERNUM_FILES)}")
    paths = []
    for pid in selected:
        expected, etag = STERNUM_FILES[pid]
        name = f"WildPPG_Part_{pid}.mat"
        final = root / name
        part = root / f"{name}.{etag}.part"
        if not final.exists():
            offset = part.stat().st_size if part.exists() else 0
            if offset > expected or shutil.disk_usage(root).free < expected - offset:
                raise OSError(f"Check free space and partial file size for {name}")
            print(f"{pid}: fetching {(expected-offset)/1e6:.0f} MB of sternum data", flush=True)
            with part.open("ab") as output:
                while offset < expected:
                    end = min(offset + 16 * 1024**2, expected) - 1
                    output.write(fetch_verified_range(WILDPPG_BASE + name, offset, end, etag))
                    offset = end + 1
            if [variable for variable, _, _ in whosmat(part, appendmat=False)] != ["id", "notes", "sternum"]:
                raise ValueError(f"Unexpected MATLAB structure in {name}; attach the original recording.")
            part.replace(final)
        if final.stat().st_size != expected:
            raise ValueError(f"Unexpected file in download cache: {final}. Use a separate empty cache directory.")
        paths.append(final)
    return paths

In [ ]:
WILDPPG_FS = 128

def find_wildppg():
    found = {}
    for root in [DATA_ROOT, WILDPPG_CACHE]:
        if root.is_dir():
            for p in root.rglob('WildPPG_Part_*.mat'):
                found.setdefault(p.stem.split('_')[-1], p)
    wanted = WILDPPG_IDS or list(STERNUM_FILES)
    missing = [i for i in wanted if i not in found]
    if missing and ALLOW_DOWNLOAD:
        for p in ensure_wildppg_inputs(WILDPPG_CACHE, missing):
            found[p.stem.split('_')[-1]] = p
    return {i: found[i] for i in wanted if i in found}

def read_sternum(path):
    mat = sio.loadmat(path, variable_names=['id', 'sternum'], simplify_cells=True)
    s = mat['sternum']
    for k in ['ppg_ir', 'acc_x', 'acc_y', 'acc_z']:
        if not np.isclose(float(s[k]['fs']), WILDPPG_FS):
            raise ValueError(f'{path.name}: unexpected {k} sampling rate')
    acc = np.vstack([np.asarray(s[k]['v'], float) for k in ['acc_x', 'acc_y', 'acc_z']])
    return (str(mat['id']).strip(), -np.asarray(s['ppg_ir']['v'], float), np.linalg.norm(acc, axis=0),
            np.asarray(s['ecg']['v'], float))

ECG_SOS = signal.butter(3, [5.0, 20.0], fs=WILDPPG_FS, btype='band', output='sos')

def ecg_beats(segment):
    # Reference beat count from the chest ECG in the same 16 s (offline check, not a detector input).
    if not np.isfinite(segment).all():
        return np.nan
    e = signal.sosfiltfilt(ECG_SOS, segment)
    e = e if np.percentile(e, 99) >= -np.percentile(e, 1) else -e
    peaks, _ = signal.find_peaks(e, distance=int(0.25 * WILDPPG_FS), prominence=0.3 * np.percentile(np.abs(e), 99))
    return len(peaks)

final_models, final_thresholds = {}, {}
all_idx = np.arange(len(y))
cv_all = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED + 99)
for m in MODELS:
    if m == 'rule_definitions':
        continue
    scores = np.full(len(y), np.nan)
    for tr, te in cv_all.split(X_feat, strata):
        scores[te] = predict(m, fit(m, tr), te)
    final_thresholds[m] = thresholds_from(scores, y)
    final_models[m] = fit(m, all_idx)
print('Final thresholds:', {m: {k: round(v, 4) for k, v in t.items()} for m, t in final_thresholds.items()})

def score_windows(waves):
    F = pd.DataFrame([ppg_features(w)[0] for w in waves])
    out = {'rule_definitions': rule_definitions(F)}
    for m, model in final_models.items():
        X = np.vstack([prepare(w) for w in waves]).astype(np.float32) if m == 'cnn_1d' else F[FEATURES].to_numpy(float)
        out[m] = model.predict_proba(X)[:, 1]
    return out

transfer_rows = []
paths = find_wildppg()
print('WildPPG participants:', list(paths))
for pid, path in paths.items():
    participant, ppg, acc, ecg = read_sternum(path)
    ppg, acc = to_device_rate(ppg, WILDPPG_FS), to_device_rate(acc, WILDPPG_FS)
    win, step = WINDOW_S * DEVICE_FS, TRANSFER_STEP_S * DEVICE_FS
    starts = [s for s in range(0, len(ppg) - win + 1, step) if np.isfinite(ppg[s:s + win]).all()]
    if not starts:
        continue
    waves = np.vstack([ppg[s:s + win] for s in starts])
    motion = np.array([np.nanstd(acc[s:s + win]) for s in starts])
    ref_beats = np.array([ecg_beats(ecg[int(s / DEVICE_FS * WILDPPG_FS):int(s / DEVICE_FS * WILDPPG_FS) + WINDOW_S * WILDPPG_FS])
                          for s in starts])
    ppg_beats = np.array([ppg_features(w)[0]['n_beats'] for w in waves])
    scores = score_windows(waves)
    for i, s in enumerate(starts):
        transfer_rows.append({'participant': pid, 'start_s': s / DEVICE_FS, 'motion_sd': motion[i],
                              'ecg_beats': ref_beats[i], 'ppg_beats': ppg_beats[i],
                              **{f'score_{m}': scores[m][i] for m in scores}})
transfer = pd.DataFrame(transfer_rows)
transfer['motion'] = pd.qcut(transfer.motion_sd, 3, labels=['low motion', 'mid motion', 'high motion'])
# Pulse visibility: PPG-detected beats / ECG-detected beats in the same window (ECG = reference instrument).
transfer['visibility'] = transfer.ppg_beats / transfer.ecg_beats.where(transfer.ecg_beats > 0)
transfer['pulse_visible'] = np.where(transfer.visibility.between(0.8, 1.2), 'PPG pulse visible (0.8-1.2 x ECG beats)',
                                     'PPG pulse not visible')
print('Median PPG/ECG beat ratio:', round(float(transfer.visibility.median()), 2),
      '| windows with visible pulse:', f"{(transfer.pulse_visible.str.startswith('PPG pulse visible')).mean():.0%}")

def start_mask(frame, flag):
    # An alert starts when a window is flagged and the previous window (same participant, 2 s earlier) was not.
    prev_same = (frame.participant.eq(frame.participant.shift()) & frame.start_s.sub(frame.start_s.shift()).eq(TRANSFER_STEP_S)).to_numpy()
    prev_flag = np.r_[False, flag[:-1]] & prev_same
    return flag & ~prev_flag

fa_rows = []
for m in MODELS:
    points = ['definition'] if m == 'rule_definitions' else ['high_sensitivity', 'high_specificity']
    for point in points:
        th = 0.5 if m == 'rule_definitions' else final_thresholds[m][point]
        flag_all = (transfer[f'score_{m}'] >= th).to_numpy()
        start_all = start_mask(transfer, flag_all)   # starts found on the full continuous sequence
        groups = [('all', np.ones(len(transfer), bool)),
                  *[(str(g), transfer.motion.eq(g).to_numpy()) for g in transfer.motion.cat.categories],
                  *[(str(g), transfer.pulse_visible.eq(g).to_numpy()) for g in sorted(transfer.pulse_visible.unique())]]
        for group, mask in groups:
            # Each alert start is attributed to the subset its first window belongs to.
            hours = mask.sum() * TRANSFER_STEP_S / 3600
            n = int(start_all[mask].sum())
            lo, hi = poisson_rate_ci(n, hours)
            fa_rows.append({'model': m, 'operating_point': point, 'subset': group,
                            'participants': transfer.participant[mask].nunique(), 'hours': hours,
                            'false_alert_starts': n, 'per_hour': n / hours, 'ci95_lo': lo, 'ci95_hi': hi,
                            'flagged_fraction': flag_all[mask].mean()})
false_alarms = pd.DataFrame(fa_rows)
display(false_alarms[false_alarms.subset.eq('all')].round(3))
display(false_alarms.pivot_table(index=['model', 'operating_point'], columns='subset', values='per_hour').round(1))

## Figures

In [ ]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# 1. Example windows: one expert-confirmed true event and one false alarm per alarm type.
fig, axes = plt.subplots(len(TARGET_ALARMS), 2, figsize=(14, 2.2 * len(TARGET_ALARMS)), sharex=True)
t = np.arange(X_wave.shape[1]) / DEVICE_FS
for row, alarm in enumerate(TARGET_ALARMS):
    for col, label in enumerate([1, 0]):
        pick = np.flatnonzero((meta.alarm == alarm).to_numpy() & (y == label))
        ax = axes[row, col]
        if len(pick):
            z = prepare(X_wave[pick[0]])
            _, peaks = ppg_features(X_wave[pick[0]])
            ax.plot(t, z, lw=.7)
            ax.plot(t[peaks], z[peaks], 'r.', ms=4)
        ax.set_title(f"{alarm}: {'true event' if label else 'false alarm'} ({meta.record.iloc[pick[0]] if len(pick) else '-'})", fontsize=8)
axes[-1, 0].set_xlabel('seconds before alarm (window end = alarm)'); axes[-1, 1].set_xlabel('seconds')
save(fig, '01_example_windows')

# 2. Sensitivity vs specificity with 95% CIs (outer cross-validation).
fig, ax = plt.subplots(figsize=(7, 5))
for r in results.itertuples():
    ax.errorbar(1 - r.specificity, r.sensitivity, xerr=[[r.spec_hi - r.specificity], [r.specificity - r.spec_lo]],
                yerr=[[r.sensitivity - r.sens_lo], [r.sens_hi - r.sensitivity]], fmt='o', capsize=2)
    ax.annotate(f'{r.model} ({r.operating_point})', (1 - r.specificity, r.sensitivity), fontsize=7)
ax.set(xlabel='1 - specificity (false monitor alarms flagged)', ylabel='sensitivity (expert-confirmed events caught)',
       xlim=(-.02, 1.02), ylim=(-.02, 1.02), title='Outer cross-validation, ICU finger PPG')
save(fig, '02_sensitivity_specificity')

# 3. False alarms on healthy chest PPG.
block = false_alarms[false_alarms.subset.eq('all')]
fig, ax = plt.subplots(figsize=(9, 4))
labels = block.model + ' / ' + block.operating_point
ax.barh(labels, block.per_hour, xerr=[block.per_hour - block.ci95_lo, block.ci95_hi - block.per_hour], capsize=2)
ax.set(xlabel='false alert starts per hour (95% CI)', title='Healthy volunteers, sternum PPG (WildPPG)')
save(fig, '03_healthy_false_alarms')

SPLIT_COLOURS = {'train': 'C0', 'valid': 'C1', 'test': 'C3'}

# 4. Training curves: class-balanced log loss and accuracy (0.5 cut-off) per boosting round / CNN epoch.
#    Line = mean over curves; band = min-max. valid = 25 inner fits; train and test = 5 outer fits.
iterative = [m for m in ['boosted_trees', 'cnn_1d'] if len(curves) and m in set(curves.model)]
if iterative:
    fig, axes = plt.subplots(len(iterative), 3, figsize=(15, 3.6 * len(iterative)), squeeze=False)
    for r, m in enumerate(iterative):
        c = curves[curves.model.eq(m)]
        for col, metric in enumerate(['loss', 'accuracy', 'balanced_accuracy']):
            ax = axes[r, col]
            for split, colour in SPLIT_COLOURS.items():
                g = c[c.split.eq(split)].groupby('step')[metric]
                mean = g.mean()
                ax.plot(mean.index, mean, color=colour, label=split)
                ax.fill_between(mean.index, g.min(), g.max(), color=colour, alpha=.15, lw=0)
            ax.set(title=f"{m}: {metric.replace('_', ' ')}", xlabel='boosting round' if m == 'boosted_trees' else 'epoch',
                   ylabel='class-balanced log loss' if metric == 'loss' else f"{metric.replace('_', ' ')} at 0.5")
        axes[r, 0].legend(fontsize=8)
    save(fig, '04_training_curves')

# 5. ROC curves. Left: pooled out-of-fold test scores (the AUROC in the results table) with the rule's fixed point.
#    Others: per-fold ROC on train / valid / test, mean over folds, band = min-max.
learned = [m for m in MODELS if m != 'rule_definitions']
fig, axes = plt.subplots(1, len(learned) + 1, figsize=(4.6 * (len(learned) + 1), 4.4), squeeze=False)
axes = axes[0]
for m in learned:
    fpr, tpr, _ = roc_curve(y, oof[m])
    axes[0].plot(fpr, tpr, label=f'{m} (AUROC {roc_auc_score(y, oof[m]):.2f})')
rule = results[results.model.eq('rule_definitions')].iloc[0]
axes[0].plot(1 - rule.specificity, rule.sensitivity, 'k*', ms=11, label='rule_definitions (fixed)')
axes[0].set_title('Test: outer folds pooled')
grid = np.linspace(0, 1, 201)
for ax, m in zip(axes[1:], learned):
    for split, colour in SPLIT_COLOURS.items():
        tprs = []
        for part in roc_parts:
            if part['model'] == m and part['split'] == split:
                fpr, tpr, _ = roc_curve(part['labels'], part['scores'])
                tprs.append(np.interp(grid, fpr, tpr))
        tprs = np.array(tprs)
        auc = split_metrics[split_metrics.model.eq(m) & split_metrics.split.eq(split)].AUROC.mean()
        ax.plot(grid, tprs.mean(0), color=colour, label=f'{split} (mean AUROC {auc:.2f})')
        ax.fill_between(grid, tprs.min(0), tprs.max(0), color=colour, alpha=.12, lw=0)
    ax.set_title(f'{m}: train / valid / test')
for ax in axes:
    ax.plot([0, 1], [0, 1], ':', color='grey', lw=.8)
    ax.set(xlabel='1 - specificity', ylabel='sensitivity', xlim=(0, 1), ylim=(0, 1.01))
    ax.legend(fontsize=7, loc='lower right')
save(fig, '05_roc_curves')

# 6. Train / valid / test summary per model: mean over outer folds, whiskers = min-max.
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric in zip(axes, ['AUROC', 'loss', 'accuracy', 'balanced_accuracy']):
    for k, (split, colour) in enumerate(SPLIT_COLOURS.items()):
        g = split_metrics[split_metrics.split.eq(split)].groupby('model')[metric]
        mean, lo, hi = (g.agg(f).reindex(MODELS).to_numpy(float) for f in ['mean', 'min', 'max'])
        ok = np.isfinite(mean)
        x = np.arange(len(MODELS))[ok] + (k - 1) * 0.27
        ax.bar(x, mean[ok], 0.27, yerr=[mean[ok] - lo[ok], hi[ok] - mean[ok]], capsize=2, color=colour, label=split)
    ax.set_xticks(np.arange(len(MODELS)), MODELS, rotation=20, ha='right', fontsize=8)
    ax.set_title({'AUROC': 'AUROC', 'loss': 'class-balanced log loss (lower = better)',
                  'accuracy': 'accuracy at 0.5', 'balanced_accuracy': 'balanced accuracy at 0.5'}[metric], fontsize=9)
axes[2].axhline((y == 0).mean(), color='grey', ls=':', lw=1, label='never fires')
axes[0].legend(fontsize=8); axes[2].legend(fontsize=7)
save(fig, '06_split_summary')

## Export

In [ ]:
configuration = {
    'purpose': 'pulse-lost / dangerous-rhythm detector from PPG', 'version': 'v4 (2026-11-09)',
    'deployable': False, 'drowning_detector': False,
    'training_data': {'dataset': 'PhysioNet/CinC Challenge 2015 training set', 'md5': C2015[2],
                      'labels': 'expert true/false alarm annotations (Challenge)', 'alarms_used': TARGET_ALARMS,
                      'alarms_excluded': EXCLUDED_ALARMS, 'records': int(len(y)), 'true_events': int(y.sum())},
    'false_alarm_data': {'dataset': 'WildPPG sternum IR PPG, healthy volunteers', 'participants': list(paths),
                         'hours': float(len(transfer) * TRANSFER_STEP_S / 3600)},
    'input': {'rate_hz': DEVICE_FS, 'window_s': WINDOW_S, 'band_hz': [0.5, 8.0], 'scaling': '5-95 percentile spread'},
    'rule_definitions': {'asystole_gap_s': ASYSTOLE_GAP_S, 'brady_bpm': BRADY_BPM, 'brady_beats': BRADY_BEATS},
    'operating_points': {'high_sensitivity': TARGET_SENSITIVITY, 'high_specificity': TARGET_SPECIFICITY},
    'final_thresholds': final_thresholds, 'features': FEATURES, 'models': MODELS, 'seed': SEED,
    'training': {'loss': 'class-weighted binary cross-entropy (balanced classes)',
                 'cnn_1d': {'epochs': 40, 'batch': 32, 'optimizer': 'Adam lr 1e-3, weight decay 1e-3'},
                 'boosted_trees': {'rounds': 200, 'learning_rate': 0.05}, 'early_stopping': False,
                 'diagnostic_splits': {'train': 'outer-training records, in-sample',
                                       'valid': 'inner-CV held-out records (choose thresholds)',
                                       'test': 'outer held-out fold'},
                 'curves_used_for_selection': False},
    'limitations': ['ICU finger PPG, not chest MAX30102 in water', 'record-level folds (no patient IDs published)',
                    'few true asystole (17) and VF (6) events with PPG', 'healthy-PPG test on land only'],
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                 'scikit_learn': sklearn.__version__},
}
(OUT / 'pulse_loss_configuration.json').write_text(json.dumps(configuration, indent=2, default=str), encoding='utf-8')
results.to_csv(OUT / 'cv_results.csv', index=False)
per_type.to_csv(OUT / 'cv_per_alarm_type.csv', index=False)
chosen.to_csv(OUT / 'fold_thresholds.csv', index=False)
false_alarms.to_csv(OUT / 'healthy_false_alarms.csv', index=False)
split_metrics.to_csv(OUT / 'split_metrics.csv', index=False)
curves.to_csv(OUT / 'training_curves.csv', index=False)
pd.DataFrame({'record': meta.record, 'alarm': meta.alarm, 'label': y, 'fold': fold_of,
              **{f'score_{m}': oof[m] for m in MODELS}}).to_csv(OUT / 'oof_predictions.csv', index=False)
joblib.dump({m: v for m, v in final_models.items() if m != 'cnn_1d'}, OUT / 'pulse_loss_feature_models.joblib')
if 'cnn_1d' in final_models:
    torch.save(final_models['cnn_1d'].net.state_dict(), OUT / 'pulse_loss_cnn_1d.pt')
print('Exported:', sorted(p.name for p in OUT.iterdir() if p.is_file()))

## Reading the results

- **Start with `rule_definitions`.** It uses the Challenge's own published thresholds, so nothing about it was tuned. A trained model must beat it on *both* sensitivity and healthy false alarms to justify itself.
- **Check the train / valid / test gap before trusting a learned model.** If train AUROC is near 1 while valid and test are much lower, the model has memorised these 411 records. Valid close to test means the thresholds chosen on valid should hold on new records *from the same ICU population*. It says nothing about the chest in water.
- **Read accuracy with its baseline.** Never firing scores 0.76, so balanced accuracy and sensitivity are the informative numbers.
- **Read intervals, not points.** There are only 17 true asystole and 6 true VF events with PPG, so per-type results for those are anecdotal.
- **The two halves answer different questions.**
  - Cross-validation asks: can PPG tell a real dangerous rhythm from a false monitor alarm?
  - The WildPPG test asks: how often would it fire on a healthy chest during daily life?

  A deployable detector needs both, measured on the wearable's own signal in water.
- **Next data:** the wearable's MAX30102 recorded alongside a chest ECG during swimming (false alarms), and HealthRing or WildPPG motion segments for artefact robustness.